# Setting

In [ ]:
"""Docling-based parser utilities for RAG workflows."""

from __future__ import annotations

import os
import re
from collections import OrderedDict
from typing import TYPE_CHECKING

# Patch transformers export for AutoProcessor before docling imports.
try:
    import transformers as _tf

    if not hasattr(_tf, "AutoProcessor"):
        from transformers.models.auto.processing_auto import AutoProcessor as _AutoProcessor

        _tf.AutoProcessor = _AutoProcessor
        if hasattr(_tf, "__all__") and isinstance(_tf.__all__, list):
            if "AutoProcessor" not in _tf.__all__:
                _tf.__all__.append("AutoProcessor")
except Exception:
    pass

try:
    from docling.datamodel.base_models import InputFormat
    from docling.datamodel.pipeline_options import (
        AcceleratorDevice,
        AcceleratorOptions,
        EasyOcrOptions,
        PdfPipelineOptions,
        RapidOcrOptions,
        TableStructureOptions,
        TesseractCliOcrOptions,
        TesseractOcrOptions,
    )
    from docling.document_converter import DocumentConverter, PdfFormatOption
except ImportError as exc:
    raise ImportError(
        "rag_parser_docling requires the 'docling' package. "
        "Install extras with 'pip install gaik[rag-parser-docling]'"
    ) from exc

if TYPE_CHECKING:
    from langchain_core.documents import Document

try:
    import torch  # type: ignore

    _HAS_TORCH = True
except Exception:
    _HAS_TORCH = False

# Optional summaries import (kept from existing behavior).
try:
    from src.summaries_images import summaries

    summaries = OrderedDict(summaries)
except Exception:
    summaries = OrderedDict()


def _torch_status() -> dict:
    """Return a dict with Torch/CUDA status."""
    info = {
        "torch_import_ok": False,
        "torch_version": None,
        "cuda_available": False,
        "cuda_device_count": 0,
        "cuda_device_name": None,
    }

    if not _HAS_TORCH:
        return info

    try:
        info["torch_import_ok"] = True
        info["torch_version"] = getattr(torch, "__version__", None)
        if torch.cuda.is_available():
            info["cuda_available"] = True
            info["cuda_device_count"] = torch.cuda.device_count()
            try:
                idx = torch.cuda.current_device()
            except Exception:
                idx = 0
            try:
                info["cuda_device_name"] = torch.cuda.get_device_name(idx)
            except Exception:
                info["cuda_device_name"] = None
    except Exception:
        pass

    return info


def pick_accelerator(verbose: bool = True) -> AcceleratorDevice:
    """
    Select the best available accelerator:
      - CUDA if a CUDA-enabled PyTorch build is present and available
      - CPU otherwise
    """
    status = _torch_status()

    if verbose:
        if status["torch_import_ok"]:
            print(
                f"[Torch] version: {status['torch_version']}, "
                f"cuda available: {status['cuda_available']}"
            )
        else:
            print("[Torch] not installed or failed to import")

    if status["cuda_available"]:
        if verbose:
            name = status["cuda_device_name"] or "Unknown NVIDIA GPU"
            print(f"Using CUDA device: {name} (devices: {status['cuda_device_count']})")
        return AcceleratorDevice.CUDA

    if verbose:
        print("CUDA not available. Using CPU.")
    return AcceleratorDevice.CPU


class DoclingRagParser:
    """Docling-based parser for RAG pipelines."""

    def __init__(
        self,
        *,
        enable_ocr: bool = True,
        ocr_engine: str = "rapidocr",
        enable_table_structure: bool = True,
        enable_formula_enrichment: bool = True,
        num_threads: int = 4,
        verbose: bool = True,
    ) -> None:
        self.summaries = summaries.copy()

        device = pick_accelerator(verbose=verbose)

        pipeline_kwargs = {
            "do_ocr": enable_ocr,
            "do_table_structure": enable_table_structure,
            "generate_picture_images": False,
            "generate_page_images": False,
            "do_formula_enrichment": enable_formula_enrichment,
            "table_structure_options": TableStructureOptions(
                kind="docling_tableformer",
                do_cell_matching=True,
            )
            if enable_table_structure
            else None,
            "accelerator_options": AcceleratorOptions(
                num_threads=num_threads,
                device=device,
            ),
        }
        if enable_ocr:
            pipeline_kwargs["ocr_options"] = _build_ocr_options(ocr_engine)

        self.pipeline_options = PdfPipelineOptions(**pipeline_kwargs)

        self.format_options = {
            InputFormat.PDF: PdfFormatOption(pipeline_options=self.pipeline_options)
        }
        self.converter = DocumentConverter(format_options=self.format_options)
        self.supported_extensions = [".pdf"]

    def replace_base64_images(self, md_text: str, summary_dict: OrderedDict) -> str:
        pattern = r"!\[.*?\]\(data:image\/png;base64,[A-Za-z0-9+/=\n]+\)"

        def replacement(_match):
            if summary_dict:
                _, value = summary_dict.popitem(last=False)
                return f"\n\n{value}\n\n"
            return "\n\n[Image removed - no summary available]\n\n"

        return re.sub(pattern, replacement, md_text)

    def convert_pdf_to_markdown(self, pdf_path: str, output_path: str | None = None) -> str:
        if output_path is None:
            pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
            output_path = os.path.join(os.path.dirname(pdf_path), f"{pdf_name}.md")

        result = self.converter.convert(pdf_path)
        markdown_text = result.document.export_to_markdown()
        markdown_text = self.replace_base64_images(markdown_text, self.summaries.copy())

        with open(output_path, "w", encoding="utf-8") as f:
            f.write(markdown_text)

        return markdown_text

    def convert_pdf_to_chunks_with_metadata(
        self, pdf_path: str
    ) -> list[Document]:
        """
        Convert PDF to chunks with metadata including document name and page numbers.
        Returns a list of LangChain Document objects with metadata.

        Note: This method uses Docling's HierarchicalChunker, which chunks by document
        structure (headings, sections, paragraphs) rather than by fixed size. Chunk
        boundaries are determined by the document's semantic structure.
        """
        print(f"Processing document with metadata extraction: {pdf_path}")
        print("Parsing document(s)...")
        result = self.converter.convert(pdf_path)
        doc = result.document
        print(f"Document parsing complete. Pages: {len(getattr(doc, 'pages', [])) or 'unknown'}")

        try:
            from docling.chunking import HierarchicalChunker
        except ImportError as exc:
            raise ImportError(
                "Docling chunking is required for this method. "
                "Install extras with 'pip install docling-core[chunking]'"
            ) from exc

        try:
            from langchain_core.documents import Document
        except ImportError as exc:
            raise ImportError(
                "rag_parser_docling requires 'langchain-core' for chunk output. "
                "Install extras with 'pip install gaik[rag-parser-docling]'"
            ) from exc

        # HierarchicalChunker chunks by document structure (headings, sections)
        # rather than by fixed size. Chunk boundaries follow semantic structure.
        chunker = HierarchicalChunker()
        document_name = os.path.splitext(os.path.basename(pdf_path))[0]
        langchain_docs: list[Document] = []
        chunk_id = 0

        for chunk in chunker.chunk(doc):
            try:
                chunk_text = chunk.text
                chunk_dict = chunk.model_dump()

                filename = document_name
                if "meta" in chunk_dict and "origin" in chunk_dict["meta"]:
                    origin_filename = chunk_dict["meta"]["origin"].get("filename")
                    if origin_filename:
                        filename = os.path.splitext(os.path.basename(origin_filename))[0]

                page_num = None
                if (
                    "meta" in chunk_dict
                    and "doc_items" in chunk_dict["meta"]
                    and chunk_dict["meta"]["doc_items"]
                    and "prov" in chunk_dict["meta"]["doc_items"][0]
                    and chunk_dict["meta"]["doc_items"][0]["prov"]
                ):
                    page_num = chunk_dict["meta"]["doc_items"][0]["prov"][0].get("page_no")

                heading = None
                if (
                    "meta" in chunk_dict
                    and "headings" in chunk_dict["meta"]
                    and chunk_dict["meta"]["headings"]
                ):
                    heading = chunk_dict["meta"]["headings"][0]

                metadata = {
                    "file_path": pdf_path,
                    "file_name": filename,
                    "page_number": page_num if page_num is not None else "Unknown",
                    "heading": heading,
                    "chunk_id": chunk_id,
                }

                langchain_docs.append(Document(page_content=chunk_text, metadata=metadata))
                chunk_id += 1

                if chunk_id <= 3:
                    print(
                        f"Chunk {chunk_id}: document='{filename}', "
                        f"page={page_num}, heading='{heading}'"
                    )

            except Exception as exc:
                print(f"Error processing chunk {chunk_id}: {exc}")
                fallback_meta = {
                    "file_path": pdf_path,
                    "file_name": document_name,
                    "page_number": "Unknown",
                    "heading": None,
                    "chunk_id": chunk_id,
                }
                langchain_docs.append(
                    Document(page_content=getattr(chunk, "text", ""), metadata=fallback_meta)
                )
                chunk_id += 1

        print(f"Created {len(langchain_docs)} chunks with metadata from {document_name}")
        return langchain_docs

    def convert_pdf_to_pages(self, pdf_path: str) -> list[Document]:
        """
        PDF를 페이지 단위로 파싱하여 LangChain Document 리스트로 반환합니다.
        각 Document는 해당 페이지의 텍스트를 content로 가지며, 
        metadata에 페이지 번호와 문서 정보를 포함합니다.
        """
        print(f"Processing document by pages: {pdf_path}")
        result = self.converter.convert(pdf_path)
        doc = result.document
        
        try:
            from langchain_core.documents import Document
        except ImportError as exc:
            raise ImportError(
                "rag_parser_docling requires 'langchain-core'. "
                "Install with 'pip install langchain-core'"
            ) from exc

        document_name = os.path.splitext(os.path.basename(pdf_path))[0]
        langchain_docs: list[Document] = []

        # Docling의 각 페이지 객체를 순회
        for page_no, page in doc.pages.items():
            # 해당 페이지의 요소들만 필터링하여 마크다운으로 변환
            # (image_mode="embedded"는 필요 시 유지)
            page_md = doc.export_to_markdown(page_no=page_no)
            
            # 이미지 베이스64 요약 교체 (기존 로직 활용)
            page_md = self.replace_base64_images(page_md, self.summaries.copy())

            metadata = {
                "file_path": pdf_path,
                "file_name": document_name,
                "page_number": page_no,
                "total_pages": len(doc.pages),
            }

            langchain_docs.append(Document(page_content=page_md, metadata=metadata))

        print(f"Created {len(langchain_docs)} page-level documents from {document_name}")
        return langchain_docs

def parse_pdf_to_markdown(pdf_path: str, output_path: str | None = None) -> str:
    """Convenience wrapper for DoclingRagParser.convert_pdf_to_markdown."""
    parser = DoclingRagParser()
    return parser.convert_pdf_to_markdown(pdf_path, output_path=output_path)


def parse_pdf_to_chunks_with_metadata(pdf_path: str) -> list[Document]:
    """
    Convenience wrapper for DoclingRagParser.convert_pdf_to_chunks_with_metadata.

    Chunks documents by structure (headings, sections) using HierarchicalChunker.
    """
    parser = DoclingRagParser()
    return parser.convert_pdf_to_chunks_with_metadata(pdf_path)

def parse_pdf_to_pages(pdf_path: str) -> list[Document]:
    """페이지 단위 파싱을 위한 편의용 래퍼 함수"""
    parser = DoclingRagParser()
    return parser.convert_pdf_to_pages(pdf_path)


def _build_ocr_options(ocr_engine: str):
    engine = (ocr_engine or "").lower()
    if engine == "tesseract":
        return TesseractOcrOptions()
    if engine == "tesseract_cli":
        return TesseractCliOcrOptions()
    if engine == "easyocr":
        return EasyOcrOptions()
    if engine == "rapidocr":
        return RapidOcrOptions()
    raise ValueError(
        "Unsupported OCR engine. Use 'tesseract_cli', 'tesseract', 'easyocr', or 'rapidocr'."
    )

# 1차 파싱

In [ ]:
filename = ""
res = parse_pdf_to_pages(pdf_path=f"./file/{filename}")
res

In [ ]:
from IPython.display import Markdown
Markdown(res[2].page_content)

In [ ]:
import pickle
with open("./docs/FWG.pkl", "wb") as f:
    pickle.dump(res, f)


In [ ]:
import pickle
with open("./docs/FWG.pkl", "rb") as f:
    loaded_text = pickle.load(f)
from IPython.display import Markdown
print(loaded_text[2].page_content)

# 분석용 Agent Setting

In [ ]:
import os
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

# Set API key (example for OpenAI)
os.environ["GROQ_API_KEY"] = ""

# Initialize the model
model = init_chat_model(
    model_provider="groq",
    model="moonshotai/kimi-k2-instruct-0905",
    temperature=0.0,
    max_tokens=5000
)

def get_global_context(PARTIAL_WHOLE_DOCUMENT, CHUNK_CONTENT):
    # 1. 시스템 프롬프트 개선: '부분적 맥락'임을 명시하고 출력 형식을 강제함
    SYSTEM_PROMPT = f"""You are an expert at analyzing document structures for information retrieval.

    <context_window>
    {PARTIAL_WHOLE_DOCUMENT}
    </context_window>

    The following <chunk> is located within the <context_window> provided above.
    <chunk>
    {CHUNK_CONTENT}
    </chunk>

    Your task: Provide a short, succinct sentence (under 50 words) that explains the specific role or location of this chunk within the provided context. 
    Focus on:
    - What specific topic or sub-section this chunk belongs to.
    - How it relates to the immediate preceding or following information in the window.

    Output only the succinct context and nothing else. Do not include labels like "Context:" or "Summary:"."""

    agent = create_agent(model=model,  system_prompt=SYSTEM_PROMPT)

    result = agent.invoke({"messages": [{"role": "user", "content": "Analyze the chunk's placement within the provided context window."}]})

    return result["messages"][-1].content.strip()

# 2차 파싱

In [ ]:
import pickle
with open("./docs/FWG.pkl", "rb") as f:
    loaded_text = pickle.load(f)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import List  # 리스트 타입을 위해 추가

# 1. 추출하고 싶은 메타데이터 구조 정의 (Pydantic)
class DocumentMeta(BaseModel):
    title: str = Field(default="Unknown Title", description="문서의 전체 제목")
    ship_numbers: List[str] = Field(default_factory=list, description="대상 선박 번호 리스트")
    product_name: str = Field(default="Unknown Product", description="대상 제품명")
    specifications: str = Field(default="No specs found", description="주요 제원 및 사양 요약")
    document_type: str = Field(default="Unknown Type", description="문서의 종류")

# 2. 메타데이터 추출 함수
def extract_global_metadata(loaded_text):
    # 앞 5페이지 추출 (데이터가 5페이지보다 적을 경우 대비)
    header_pages = loaded_text[:5]
    header_content = "\n".join([doc.page_content for doc in header_pages])
    
    parser = JsonOutputParser(pydantic_object=DocumentMeta)
    
    SYSTEM_PROMPT = """You are a professional maritime document analyzer. 
    Based on the first few pages of the technical specification, extract key metadata.
    
    CRITICAL INSTRUCTION: 
    - For 'ship_numbers', identify all individual ship/hull numbers (e.g., '8250/8251' should be ['8250', '8251']).
    - Ensure each number is a separate string in the list.
    - Respond only in JSON format matching the given schema."""
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("user", "Extract metadata from the following text:\n\n{content}\n\n{format_instructions}")
    ]).partial(format_instructions=parser.get_format_instructions())

    # 모델 호출 (여기서는 고성능 모델 권장)
    chain = prompt | model | parser
    return chain.invoke({"content": header_content})

In [ ]:
glabal_meta = extract_global_metadata(loaded_text)
glabal_meta

In [ ]:
import json
from tqdm import tqdm

# 설정: 앞뒤로 참조할 페이지 수
WINDOW_SIZE = 5 

for i in tqdm(range(len(loaded_text))):
    # 1. 현재 처리할 페이지(Chunk) 선택
    current_doc = loaded_text[i]
    CHUNK_CONTENT = current_doc.page_content
    
    # 2. 앞뒤 5페이지 범위 계산 (0보다 작거나 전체 길이를 넘지 않도록 제한)
    start_idx = max(0, i - WINDOW_SIZE)
    end_idx = min(len(loaded_text), i + WINDOW_SIZE + 1)
    
    # 3. 해당 범위의 페이지들만 합쳐서 '부분적 전체 문서' 생성
    # Anthropic 기법에서는 이 범위가 해당 청크의 "전체 맥락" 역할을 합니다.
    context_window_docs = loaded_text[start_idx:end_idx]
    PARTIAL_WHOLE_DOCUMENT = "\n".join(doc.page_content for doc in context_window_docs)
    
    # 4. 맥락 생성 함수 호출 (기존 get_global_context 활용)
    # 팁: 프롬프트에 "이 내용은 문서의 일부(window)입니다"라고 명시하면 더 정확합니다.
    global_context = get_global_context(PARTIAL_WHOLE_DOCUMENT, CHUNK_CONTENT)
    
    # 5. 결과 업데이트
    metadata_str = json.dumps(glabal_meta, ensure_ascii=False)
    current_doc.page_content = f"Global Metadata: {metadata_str} \n\n Global Context: {global_context} \n\n Content: {CHUNK_CONTENT}"
    current_doc.metadata["global_metadata"] = glabal_meta
    current_doc.metadata["context_window_range"] = f"{start_idx}-{end_idx-1}"

In [ ]:
loaded_text

In [ ]:
from IPython.display import Markdown
print(loaded_text[2].page_content)

In [ ]:
import pickle
with open("./docs/FWG_with_global.pkl", "wb") as f:
    pickle.dump(loaded_text, f)

In [ ]:
import pickle
with open("./docs/FWG_with_global.pkl", "rb") as f:
    loaded_text2 = pickle.load(f)

In [ ]:
loaded_text2

In [ ]:
Markdown(loaded_text2[2].page_content)